# Phần 2 — Vòng lặp huấn luyện và So sánh

**Mục tiêu:** Tự viết training loop và huấn luyện 4 mô hình từ Phần 1.

Theo yêu cầu bài tập: **Không dùng** trainer.fit() hay API cấp cao.
Tự viết từng bước:
1. `optimizer.zero_grad()` — Xoá gradient cũ
2. `logits = model(x)` — Forward pass
3. `loss = criterion(logits, y)` — Tính loss
4. `loss.backward()` — Backpropagation
5. `clip_grad_norm_(...)` — Clip gradient
6. `optimizer.step()` — Cập nhật tham số

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import json
import matplotlib.pyplot as plt

from src.data import get_cifar100_loaders, get_device
from src.models_part1 import SoftmaxRegression, MLP, SimpleCNN, SimpleViT
from src.train import fit, evaluate, load_best_model
from src.utils import (
    get_param_count, compute_metrics, get_predictions,
    plot_training_curves, plot_multi_curves, plot_comparison_bar,
    print_results_table, save_metrics_json
)

DEVICE = get_device()
print(f"Thiết bị: {DEVICE}")

os.makedirs('../exercise/results/checkpoints', exist_ok=True)
os.makedirs('../exercise/results/plots', exist_ok=True)
os.makedirs('../exercise/results/metrics', exist_ok=True)

In [ ]:
# Tải dataset
train_loader, val_loader, test_loader, class_names = get_cifar100_loaders(batch_size=128)

## 1. Vòng lặp huấn luyện — Giải thích từng bước

```python
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()  # Bật dropout + batchnorm training mode

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Bước 1: Xoá gradient
        # PyTorch CỘNG DỒN gradient qua các iterations
        # → Phải xoá trước mỗi batch, không phải cuối epoch!
        optimizer.zero_grad()

        # Bước 2: Forward pass
        logits = model(x)   # [B, 100]

        # Bước 3: Tính loss
        # CrossEntropyLoss = log_softmax(logits) + NLLLoss
        # = -log(softmax(logits)[correct_class])
        loss = criterion(logits, y)

        # Bước 4: Backward pass
        # Tính đạo hàm ∂loss/∂θ cho mọi tham số θ
        loss.backward()

        # Bước 5: Gradient clipping
        # Nếu norm của gradient > 1.0, scale xuống
        # Ngăn "exploding gradients" đặc biệt trong RNN/Transformer
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Bước 6: Cập nhật tham số
        # θ = θ - lr × ∂loss/∂θ  (với AdamW thì phức tạp hơn một chút)
        optimizer.step()
```

**Scheduler:** `CosineAnnealingLR` — Learning rate giảm dần theo hình cosine:
```
LR(t) = lr_min + 0.5 × (lr_max - lr_min) × (1 + cos(π × t/T_max))
```
→ Bắt đầu cao (learn nhanh), kết thúc thấp (tinh chỉnh)

## 2. Cấu hình Hyperparameters

| Mô hình | LR | Batch | Epochs | Lý do |
|---------|-----|-------|--------|-------|
| SoftmaxRegression | 0.1 | 256 | 30 | Bài toán lồi, LR cao hội tụ nhanh |
| MLP | 1e-3 | 128 | 50 | LR thấp hơn vì có nhiều lớp |
| SimpleCNN | 1e-3 | 128 | 50 | Standard CNN setup |
| SimpleViT | 3e-4 | 128 | 100 | ViT cần nhiều epochs hơn CNN khi train từ đầu |

**Kỳ vọng accuracy** (CIFAR-100, không dùng pretrained):
- Softmax: ~15-20% (chỉ tuyến tính)
- MLP: ~35-42% (phi tuyến nhưng không có spatial bias)
- CNN: ~50-60% (tận dụng cấu trúc không gian)
- ViT: ~35-50% (cần nhiều data hơn để cạnh tranh với CNN)

## 3. Huấn luyện các mô hình

⚠️ **Lưu ý:** Quá trình train có thể mất vài giờ trên CPU.
Trên GPU/MPS thường nhanh hơn 5-10×.

Đặt `TRAIN_MODE = False` để tải checkpoints đã train sẵn (nếu có).

In [ ]:
TRAIN_MODE = True  # Đặt False để load checkpoint đã train

histories = {}

# ── Softmax Regression ──────────────────────────────────────────────
model = SoftmaxRegression(num_classes=100).to(DEVICE)
ckpt = '../exercise/results/checkpoints/softmax.pt'
config = {"epochs": 30, "lr": 0.1, "device": DEVICE, "save_path": ckpt}

if TRAIN_MODE:
    print("=" * 50)
    print("Training: SoftmaxRegression")
    print("=" * 50)
    histories["SoftmaxRegression"] = fit(model, train_loader, val_loader, config)
    save_metrics_json(histories["SoftmaxRegression"],
                      '../exercise/results/metrics/softmax_history.json')
else:
    from src.utils import load_metrics_json
    histories["SoftmaxRegression"] = load_metrics_json('../exercise/results/metrics/softmax_history.json')
    model = load_best_model(model, ckpt, DEVICE)

print("✓ SoftmaxRegression done")

In [ ]:
# ── MLP ──────────────────────────────────────────────────────────────
model_mlp = MLP(num_classes=100).to(DEVICE)
ckpt_mlp = '../exercise/results/checkpoints/mlp.pt'
config_mlp = {"epochs": 50, "lr": 1e-3, "device": DEVICE, "save_path": ckpt_mlp}

if TRAIN_MODE:
    print("=" * 50)
    print("Training: MLP")
    print("=" * 50)
    histories["MLP"] = fit(model_mlp, train_loader, val_loader, config_mlp)
    save_metrics_json(histories["MLP"], '../exercise/results/metrics/mlp_history.json')
else:
    histories["MLP"] = load_metrics_json('../exercise/results/metrics/mlp_history.json')
    model_mlp = load_best_model(model_mlp, ckpt_mlp, DEVICE)

print("✓ MLP done")

In [ ]:
# ── SimpleCNN ────────────────────────────────────────────────────────
model_cnn = SimpleCNN(num_classes=100).to(DEVICE)
ckpt_cnn = '../exercise/results/checkpoints/cnn.pt'
config_cnn = {"epochs": 50, "lr": 1e-3, "device": DEVICE, "save_path": ckpt_cnn}

if TRAIN_MODE:
    print("=" * 50)
    print("Training: SimpleCNN")
    print("=" * 50)
    histories["SimpleCNN"] = fit(model_cnn, train_loader, val_loader, config_cnn)
    save_metrics_json(histories["SimpleCNN"], '../exercise/results/metrics/cnn_history.json')
else:
    histories["SimpleCNN"] = load_metrics_json('../exercise/results/metrics/cnn_history.json')
    model_cnn = load_best_model(model_cnn, ckpt_cnn, DEVICE)

print("✓ SimpleCNN done")

In [ ]:
# ── SimpleViT ────────────────────────────────────────────────────────
model_vit = SimpleViT(num_classes=100).to(DEVICE)
ckpt_vit = '../exercise/results/checkpoints/vit.pt'
config_vit = {"epochs": 100, "lr": 3e-4, "device": DEVICE, "save_path": ckpt_vit}

if TRAIN_MODE:
    print("=" * 50)
    print("Training: SimpleViT (PyTorch)")
    print("=" * 50)
    histories["SimpleViT"] = fit(model_vit, train_loader, val_loader, config_vit)
    save_metrics_json(histories["SimpleViT"], '../exercise/results/metrics/vit_history.json')
else:
    histories["SimpleViT"] = load_metrics_json('../exercise/results/metrics/vit_history.json')
    model_vit = load_best_model(model_vit, ckpt_vit, DEVICE)

print("✓ SimpleViT done")

## 4. Biểu đồ Training Curves

In [ ]:
# Vẽ training curves cho từng mô hình riêng lẻ
for name, hist in histories.items():
    plot_training_curves(hist, title=name,
                         save_path=f'../exercise/results/plots/{name.lower()}_curves.png')

In [ ]:
# Vẽ val_accuracy của tất cả mô hình trên 1 biểu đồ
plot_multi_curves(
    list(histories.values()),
    list(histories.keys()),
    title="So sánh Val Accuracy — 4 mô hình (Phần 1)",
    save_path='../exercise/results/plots/part1_2_comparison_curves.png'
)

## 5. Đánh giá trên tập Test

In [ ]:
# Tải lại checkpoint tốt nhất và evaluate trên test set
trained_models = {
    "SoftmaxRegression": (SoftmaxRegression(100), '../exercise/results/checkpoints/softmax.pt'),
    "MLP": (MLP(100), '../exercise/results/checkpoints/mlp.pt'),
    "SimpleCNN": (SimpleCNN(100), '../exercise/results/checkpoints/cnn.pt'),
    "SimpleViT": (SimpleViT(100), '../exercise/results/checkpoints/vit.pt'),
}

results = {}
for name, (m, ckpt_path) in trained_models.items():
    if os.path.exists(ckpt_path):
        m = load_best_model(m, ckpt_path, DEVICE)
        preds, labels = get_predictions(m, test_loader, DEVICE)
        metrics = compute_metrics(preds, labels)

        # Val acc từ history
        best_val_acc = max(histories[name]["val_acc"])

        results[name] = {
            "test_acc": metrics["accuracy"],
            "val_acc": best_val_acc,
            "f1_macro": metrics["f1_macro"],
            "params": get_param_count(m),
        }

save_metrics_json(results, '../exercise/results/metrics/part1_2_results.json')
print_results_table(results)

In [ ]:
# Biểu đồ so sánh test accuracy
plot_comparison_bar(
    results, metric="test_acc",
    title="Test Accuracy — Phần 1 & 2 (CIFAR-100)",
    save_path='../exercise/results/plots/part1_2_bar.png'
)

## 6. Nhận xét và Phân tích

**Phân tích kết quả:**

1. **Softmax Regression** đạt accuracy thấp nhất vì:
   - Chỉ học được **biên quyết định tuyến tính**
   - Không thể phân biệt các lớp phức tạp với đặc trưng phi tuyến
   - 100 lớp với chỉ phép ánh xạ tuyến tính → bị underfitting nghiêm trọng

2. **MLP** tốt hơn nhờ **lớp ẩn phi tuyến**, nhưng:
   - `Flatten` làm mất **thông tin không gian** (spatial)
   - Pixel (0,0) và (1,1) kề nhau được xử lý như 2 features độc lập hoàn toàn
   - Dễ **overfitting** với 100 lớp và ít data

3. **CNN** thường đạt tốt nhất nhờ:
   - **Tích chập cục bộ** (local connectivity): học đặc trưng từ vùng lân cận
   - **Weight sharing**: giảm số params, giảm overfitting
   - **Translation invariance**: nhận diện bất kể vị trí
   - Phù hợp với **inductive bias** của ảnh (spatial structure quan trọng)

4. **ViT** có thể kém CNN khi:
   - Train từ đầu với dataset nhỏ (~45K ảnh)
   - Không có spatial inductive bias → cần học hoàn toàn từ data
   - Thường cần **pretraining** trên dataset lớn (như ImageNet) để phát huy hết sức mạnh

**Kết luận:** Với dataset nhỏ và train từ đầu, CNN thường vượt trội ViT.
ViT chỉ thực sự mạnh khi được pretrain trên hàng triệu ảnh.